In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import pyterrier_deepct
import pyterrier as pt
if not pt.started():
  pt.init() 
path = '/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/'
deepct = pyterrier_deepct.DeepCTTransformer(path + "bert_files/bert-base-uncased/bert_config.json", "/workspace/src/evaluate_deepct/marco/model.ckpt-65816", vocab_file = path + "bert_files/bert-base-uncased/vocab.txt", max_seq_length=512)
import pandas as pd
path = '/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/'
splitted_df = pd.read_csv(path + 'splitted_df.csv')
splitted_df = splitted_df.iloc[: , 1:]
splitted_df['text'] = splitted_df['text'].astype(str)
splitted_df['docno'] = splitted_df['docno'].astype(str)

In [ ]:
splits = np.array_split(splitted_df, 3)
splits_transformed = []
for split in splits:
    splits_transformed.append(deepct.transform(split))

In [ ]:
concat = pd.concat(splits_transformed)
transformed = concat.copy()
transformed['text'] = concat.groupby(['docno'])['text'].transform(lambda x : ' '.join(x))
transformed = transformed.drop_duplicates()
transformed

In [ ]:
from xml.etree import ElementTree
tree = ElementTree.parse('/workspace/src/evaluate_targers/topics2020.xml')
data = tree.getroot()
titles = []
title_ids = []
nums = []
for topic in data:
    titles.append(topic.find('title').text.lower()[:-1])
    num = topic.find('number').text
    nums.append(num)
tree = ElementTree.parse('/workspace/src/evaluate_targers/topics.xml')
data = tree.getroot()    
for topic in data:
    titles.append(topic.find('title').text.lower()[:-1])
    num = topic.find('number').text
    nums.append(num)
topics = pd.DataFrame()
topics['qid'] = nums
topics['query'] = titles

qrels_path=("/workspace/src/evaluate_targers/qrels.qrels")
qrels = pt.io.read_qrels(qrels_path)

docnos = qrels.docno

In [ ]:
save_path = '/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/'
# df_test = pd.read_csv(save_path + 'topics_conclusions_transformed.csv')

path_targer = '/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/targer_indexes/'
pd_indexer = pt.DFIndexer(path_targer + "pd_index_deepct-marco")
indexref = pd_indexer.index(transformed["text"], transformed["docno"])
# indexref = pt.IndexRef.of("/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/targer_indexes/pd_index_deepct-conclusions1/data.properties")

index = pt.IndexFactory.of(indexref)
retr = pt.BatchRetrieve(index, controls = {"wmodel": "TF_IDF"})
res=retr.transform(topics)

res

In [ ]:
eval = pt.Utils.evaluate(res,qrels,metrics=["ndcg_cut_5", "bpref", "ndcg_cut_25"])
eval

In [ ]:
res_copy = res.copy()
res_copy = res_copy[res_copy['docno'].isin(docnos)]
res_copy.head(10)

eval = pt.Utils.evaluate(res_copy,qrels,metrics=["ndcg_cut_5", "bpref", "ndcg_cut_25"])
eval